# Permutation Flow-Shop Scheduling Problem - Benchmark

This notebook solves a harder benchmark instance of the permutation
flow-shop scheduling problem (PFSSP): Taillard's instance `ta017`, with
20 jobs and 10 machines (optimal makespan 1484). See the
[PFSSP tutorial](pfssp.ipynb) for the problem formulation and the
lower bounds used.

Instances are available in the `data/flow-shop` folder.

In [ ]:
import json
import time

from bnbprob.pafssp import (
    CallbackBnB,
    CycleBestFlowShop,
    PermFlowShop,
    plot_gantt,
)

## Load Instance

Processing times of each job (row) on each machine (column).

In [ ]:
with open("../data/flow-shop/ta017.json", mode="r", encoding="utf8") as f:
    p = json.load(f)

print(f"{len(p)} jobs; {len(p[0])} machines")

## Solve

The warmstart uses the Iterated Greedy Algorithm (`constructive='iga'`)
and `CallbackBnB` applies a local search to every new incumbent.

The node manager `CycleBestFlowShop` is a cyclic best-first search over
the levels of the tree. With a negative `max_size` it starts in its
fallback mode (deepest non-empty level first), and with
`permanent_fallback=True` it never leaves it: a depth-first search in
which nodes of the same level are dequeued by best bound (and least
idle time as a tie-breaker).

Timings are illustrative and depend on the hardware.

In [ ]:
problem = PermFlowShop.from_p(p, constructive='iga')
bnb = CallbackBnB(problem)
bnb.set_manager(CycleBestFlowShop(-1, permanent_fallback=True))

start = time.time()
sol = bnb.solve(maxiter=100_000_000, timelimit=3600)
end = time.time()
print(f"Solved in {end - start:.3f} s ({bnb.explored} nodes explored)")
print(sol)

## Visualize

In [ ]:
plot_gantt(sol.problem.sequence, dpi=120, seed=42, figsize=[8, 3])

## MILP

For comparison, the positional MILP model can be solved by HiGHS
(requires `pyomo` and `highspy`):

```python
import pyomo.environ as pyo

from bnbprob.pafssp.mip import positional_model

model = positional_model(p)
solver = pyo.SolverFactory('appsi_highs')
solver.options['mip_heuristic_effort'] = 0.1
solver.options['time_limit'] = 600
solver.solve(model, tee=True)
```